# 1장 3강: 데이터 특성에 따른 가설검정 기법 선택 — 실습문제

## 실습 목표

- 비교할 변수의 척도와 집단 관계를 확인할 수 있다.
- Shapiro-Wilk 검정으로 정규성을 확인할 수 있다.
- Levene 검정으로 등분산성을 확인할 수 있다.
- 가정 점검 결과에 따라 독립표본 t검정과 Welch t검정을 선택할 수 있다.
- 선택한 검정의 p-value를 해석하여 데이터에 근거한 결론을 작성할 수 있다.

## 실습 환경 / 데이터

- Python
- pandas
- scipy.stats
- `ames_housing.csv`

| 컬럼 | 의미 | 유형 |
|---|---|---|
| `SalePrice` | 주택 판매가격 | 연속형 |
| `KitchenQual` | 주방 품질 | 범주형·순서형 |
| `HeatingQC` | 난방 품질 | 범주형·순서형 |

> 모든 판단의 유의수준은 `α = 0.05`입니다.  
> 표본 추출에는 `random_state=42`를 사용하여 실행할 때마다 같은 결과가 나오도록 합니다.


## 실습 준비

1. pandas와 `scipy.stats`를 불러오세요.
2. Ames Housing 데이터를 `df`에 불러오세요.
3. 데이터 크기, 전체 결측치 수, 컬럼명과 상위 5개 행을 확인하세요.

In [1]:
# [실습 준비] 라이브러리 임포트 및 데이터 무결성 검증
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

# 1. 파일 상대 경로 설정 (주피터 노트북 .ipynb 및 .py 완벽 호환)
try:
    ROOT = Path(__file__).resolve().parent
except NameError:
    ROOT = Path.cwd()

data_path = ROOT / "ames_housing.csv"
if not data_path.exists():
    data_path = ROOT / "data" / "ames_housing.csv"

# 2. 데이터 불러오기 (한글 깨짐 방지)
df = pd.read_csv(data_path, encoding="utf-8-sig")

# 3. 데이터 크기, 결측치, 컬럼 확인
print("=" * 60)
print("[데이터 기본 정보]")
print(f"- 데이터 형상 (Rows, Cols) : {df.shape}")
print(f"- 전체 결측치 수 (Total NA): {df.isnull().sum().sum()} 건")
print(
    f"- 주요 컬럼 확인: SalePrice, KitchenQual, HeatingQC 포함 여부 -> "
    f"{all(c in df.columns for c in ['SalePrice', 'KitchenQual', 'HeatingQC'])}"
)
print("=" * 60)
df[["SalePrice", "KitchenQual", "HeatingQC"]].head()

[데이터 기본 정보]
- 데이터 형상 (Rows, Cols) : (1460, 10)
- 전체 결측치 수 (Total NA): 0 건
- 주요 컬럼 확인: SalePrice, KitchenQual, HeatingQC 포함 여부 -> True


,SalePrice,KitchenQual,HeatingQC
0,208500,Gd,Ex
1,181500,TA,Ex
2,223500,Gd,Ex
3,140000,Gd,Gd
4,250000,Gd,Ex


---

## 필수 1. 정규성은 충족하지만 등분산성이 위반된 경우

### 문제 1-1. 주방 품질 `Gd`와 `TA` 집단의 판매가격 비교

#### 문제 설명

주방 품질이 `Gd`(Good)인 주택과 `TA`(Typical/Average)인 주택의 판매가격을 비교하려고 합니다. 각 집단에서 30개씩 표본을 추출한 뒤 정규성과 등분산성을 확인하고 적절한 t검정 방법을 선택하세요.

#### 요구사항

1. `KitchenQual == "Gd"`인 주택의 `SalePrice`에서 30개를 추출해 `group_gd`에 저장하세요.
2. `KitchenQual == "TA"`인 주택의 `SalePrice`에서 30개를 추출해 `group_ta`에 저장하세요.
3. 두 집단의 표본 수와 평균을 확인하세요.
4. 각 집단에 Shapiro-Wilk 정규성 검정을 수행하세요.
5. 두 집단에 Levene 등분산 검정을 수행하세요.
6. 각 가정의 p-value를 0.05와 비교하여 충족 여부를 판단하세요.
7. 다음 규칙에 따라 t검정 방법을 선택하세요.
   - 두 집단 모두 정규성 충족 + 등분산성 충족: 독립표본 t검정
   - 두 집단 모두 정규성 충족 + 등분산성 위반: Welch t검정
8. 선택한 검정을 실행하고 검정통계량과 p-value를 출력하세요.
9. 두 집단의 판매가격 차이가 통계적으로 유의한지 해석하세요.

#### 해석 질문

**Q1.** Shapiro-Wilk 검정의 귀무가설은 무엇인가요?  
**Q2.** Levene 검정의 귀무가설은 무엇인가요?  
**Q3.** 가정 점검 결과 어떤 t검정 방법을 선택해야 하나요?  
**Q4.** 선택한 검정의 결과에 따르면 두 집단의 판매가격 차이는 유의한가요?

#### 제출 결과

- 집단별 표본 수와 평균
- 정규성 및 등분산성 검정 결과
- t검정 방법 선택과 선택 근거
- 최종 검정통계량과 p-value
- 결과 해석
- Q1~Q4 답변


In [2]:
# [필수 1] 주방 품질 'Gd'와 'TA'의 판매가격 비교 (정규성 충족 + 등분산성 위반 케이스)

# 1 & 2. 30개씩 무작위 표본 추출 (random_state=42)
group_gd = (
    df[df["KitchenQual"] == "Gd"]["SalePrice"]
    .dropna()
    .sample(n=30, random_state=42)
)
group_ta = (
    df[df["KitchenQual"] == "TA"]["SalePrice"]
    .dropna()
    .sample(n=30, random_state=42)
)

# 3. 표본 수 및 평균 확인
n_gd, mean_gd = len(group_gd), group_gd.mean()
n_ta, mean_ta = len(group_ta), group_ta.mean()

# 4. 정규성 검정 (Shapiro-Wilk)
stat_s_gd, p_s_gd = stats.shapiro(group_gd)
stat_s_ta, p_s_ta = stats.shapiro(group_ta)

# 5. 등분산성 검정 (Levene)
stat_l, p_l = stats.levene(group_gd, group_ta)

# 6 & 7. 가설 충족 여부 판단 및 t-검정 선택
norm_gd = p_s_gd >= 0.05
norm_ta = p_s_ta >= 0.05
equal_variance = p_l >= 0.05

if norm_gd and norm_ta:
    if equal_variance:
        test_method = "독립표본 t-검정 (Student's t-test, equal_var=True)"
        t_stat, p_val = stats.ttest_ind(group_gd, group_ta, equal_var=True)
    else:
        test_method = "웰치 t-검정 (Welch's t-test, equal_var=False)"
        t_stat, p_val = stats.ttest_ind(group_gd, group_ta, equal_var=False)
else:
    test_method = (
        "정규성 위반 (비모수 검정 고려 필요)"  # 본 문제에서는 정규성 충족 조건
    )
    t_stat, p_val = stats.ttest_ind(group_gd, group_ta, equal_var=False)

# 8 & 9. 결과 출력
print("=" * 65)
print("[필수 1: KitchenQual 'Gd' vs 'TA' 분석 결과]")
print("=" * 65)
print(f"1. 표본 통계치:")
print(f"  - Gd (Good)   : n = {n_gd}, 표본평균 = ${mean_gd:,.2f}")
print(f"  - TA (Typical): n = {n_ta}, 표본평균 = ${mean_ta:,.2f}")
print(f"  - 평균 차이   : ${mean_gd - mean_ta:,.2f}")
print("-" * 65)
print(f"2. 사전 가정 검정 (α = 0.05):")
print(
    f"  - Shapiro-Wilk (Gd) : W = {stat_s_gd:.4f}, p = {p_s_gd:.4f} -> {'충족' if norm_gd else '위반'}"
)
print(
    f"  - Shapiro-Wilk (TA) : W = {stat_s_ta:.4f}, p = {p_s_ta:.4f} -> {'충족' if norm_ta else '위반'}"
)
print(
    f"  - Levene 등분산성   : F = {stat_l:.4f}, p = {p_l:.4f} -> {'등분산 충족' if equal_variance else '등분산 위반'}"
)
print("-" * 65)
print(f"3. 검정 기법 선택 및 최종 결과:")
print(f"  - 선택된 검정 기법: {test_method}")
print(f"  - t-통계량 (t-statistic): {t_stat:.4f}")
print(f"  - p-값 (p-value)        : {p_val:.4e}")
print(
    f"  - 통계적 유의성 판정    : {'[유의한 차이 있음] (p < 0.05)' if p_val < 0.05 else '[유의한 차이 없음]'}"
)
print("=" * 65)

[필수 1: KitchenQual 'Gd' vs 'TA' 분석 결과]
1. 표본 통계치:
  - Gd (Good)   : n = 30, 표본평균 = $190,591.07
  - TA (Typical): n = 30, 표본평균 = $133,586.67
  - 평균 차이   : $57,004.40
-----------------------------------------------------------------
2. 사전 가정 검정 (α = 0.05):
  - Shapiro-Wilk (Gd) : W = 0.9348, p = 0.0661 -> 충족
  - Shapiro-Wilk (TA) : W = 0.9669, p = 0.4571 -> 충족
  - Levene 등분산성   : F = 6.0628, p = 0.0168 -> 등분산 위반
-----------------------------------------------------------------
3. 검정 기법 선택 및 최종 결과:
  - 선택된 검정 기법: 웰치 t-검정 (Welch's t-test, equal_var=False)
  - t-통계량 (t-statistic): 4.4170
  - p-값 (p-value)        : 5.4796e-05
  - 통계적 유의성 판정    : [유의한 차이 있음] (p < 0.05)


### 필수 1 답변 작성란

- **Q1. Shapiro-Wilk 검정의 귀무가설은 무엇인가요?**
  * **답변:** **"데이터(표본)가 정규분포를 따른다"**입니다. 따라서 $p$-value가 유의수준 0.05 이상($p \ge 0.05$)이어야 귀무가설을 기각하지 못하여 정규성 가정을 충족하게 됩니다.
- **Q2. Levene 검정의 귀무가설은 무엇인가요?**
  * **답변:** **"비교하는 두 집단의 분산이 동일하다 (등분산이다)"**입니다. $p$-value가 0.05 미만($p < 0.05$)으로 나오면 귀무가설이 기각되어 등분산성 가정이 위반(이분산)된 것으로 판단합니다.
- **Q3. 가정 점검 결과 어떤 t검정 방법을 선택해야 하나요?**
  * **답변:** 두 집단 모두 Shapiro-Wilk 검정에서 $p \ge 0.05$로 정규성은 만족하지만, Levene 검정 결과 $p < 0.05$로 등분산성이 위반되었으므로 **웰치 t-검정(Welch's t-test, `equal_var=False`)**을 선택해야 합니다.
- **Q4. 선택한 검정의 결과에 따르면 두 집단의 판매가격 차이는 유의한가요?**
  * **답변:** **네, 통계적으로 매우 유의합니다.** Welch's t-검정 결과 $p$-value가 $0.05$보다 극단적으로 작게 산출되므로($p < 0.001$), 주방 품질이 'Gd'인 주택과 'TA'인 주택의 평균 판매가격 차이는 단순 표본 오차에 의한 우연이 아닌 유의미한 차이라고 결론 내립니다.

---

## 필수 2. 정규성과 등분산성이 모두 충족된 경우

### 문제 2-1. 난방 품질 `TA`와 `Fa` 집단의 판매가격 비교

#### 문제 설명

난방 품질이 `TA`(Typical/Average)인 주택과 `Fa`(Fair)인 주택의 판매가격을 비교하려고 합니다. 각 집단에서 20개씩 표본을 추출한 뒤 정규성과 등분산성을 확인하고 적절한 t검정 방법을 선택하세요.

#### 요구사항

1. `HeatingQC == "TA"`와 `HeatingQC == "Fa"`인 집단에서 `SalePrice`를 20개씩 추출하세요.
2. 두 집단의 표본 수와 평균을 확인하세요.
3. 두 집단이 독립집단인지 대응집단인지 판단하세요.
4. 각 집단에 Shapiro-Wilk 정규성 검정을 수행하세요.
5. 두 집단에 Levene 등분산 검정을 수행하세요.
6. 두 집단 모두 정규성을 충족하는지 확인하세요.
7. 등분산성 결과에 따라 독립표본 t검정 또는 Welch t검정 중 적절한 방법을 선택하세요.
8. 선택한 검정을 실행하고 검정통계량과 p-value를 출력하세요.
9. 두 집단의 판매가격 차이가 통계적으로 유의한지 해석하세요.

#### 해석 질문

**Q1.** 두 집단은 독립집단인가요, 대응집단인가요?  
**Q2.** 두 집단의 정규성 가정은 충족되나요?  
**Q3.** 두 집단의 등분산성 가정은 충족되나요?  
**Q4.** 가정 점검 결과 어떤 t검정 방법을 선택해야 하나요?  
**Q5.** 최종 검정 결과는 무엇을 의미하나요?

#### 제출 결과

- 집단별 표본 수와 평균
- 집단 관계 판단
- 정규성 및 등분산성 검정 결과
- t검정 방법과 선택 근거
- 검정통계량과 p-value
- 결과 해석
- Q1~Q5 답변


In [3]:
# [필수 2] 난방 품질 'TA'와 'Fa'의 판매가격 비교 (정규성 충족 + 등분산성 충족 케이스)

# 1. 20개씩 무작위 추출 (random_state=42)
group_ta_heat = (
    df[df["HeatingQC"] == "TA"]["SalePrice"]
    .dropna()
    .sample(n=20, random_state=42)
)
group_fa_heat = (
    df[df["HeatingQC"] == "Fa"]["SalePrice"]
    .dropna()
    .sample(n=20, random_state=42)
)

# 2. 표본 통계
n_ta_h, mean_ta_h = len(group_ta_heat), group_ta_heat.mean()
n_fa_h, mean_fa_h = len(group_fa_heat), group_fa_heat.mean()

# 4. 정규성 검정 (Shapiro-Wilk)
stat_s_ta_h, p_s_ta_h = stats.shapiro(group_ta_heat)
stat_s_fa_h, p_s_fa_h = stats.shapiro(group_fa_heat)

# 5. 등분산성 검정 (Levene)
stat_l_h, p_l_h = stats.levene(group_ta_heat, group_fa_heat)

# 6 & 7. 검정 선택
norm_ta_h = p_s_ta_h >= 0.05
norm_fa_h = p_s_fa_h >= 0.05
equal_var_h = p_l_h >= 0.05

if norm_ta_h and norm_fa_h:
    if equal_var_h:
        test_method_2 = "독립표본 t-검정 (Student's t-test, equal_var=True)"
        t_stat_2, p_val_2 = stats.ttest_ind(
            group_ta_heat, group_fa_heat, equal_var=True
        )
    else:
        test_method_2 = "웰치 t-검정 (Welch's t-test, equal_var=False)"
        t_stat_2, p_val_2 = stats.ttest_ind(
            group_ta_heat, group_fa_heat, equal_var=False
        )
else:
    test_method_2 = "정규성 위반 (비모수 검정 고려)"
    t_stat_2, p_val_2 = stats.ttest_ind(
        group_ta_heat, group_fa_heat, equal_var=False
    )

# 8 & 9. 출력
print("=" * 65)
print("[필수 2: HeatingQC 'TA' vs 'Fa' 분석 결과]")
print("=" * 65)
print(f"1. 집단 관계: 서로 다른 주택들이므로 [독립표본 (Independent)]")
print(f"2. 표본 통계치:")
print(f"  - TA (Typical): n = {n_ta_h}, 표본평균 = ${mean_ta_h:,.2f}")
print(f"  - Fa (Fair)   : n = {n_fa_h}, 표본평균 = ${mean_fa_h:,.2f}")
print(f"  - 평균 차이   : ${mean_ta_h - mean_fa_h:,.2f}")
print("-" * 65)
print(f"3. 사전 가정 검정 (α = 0.05):")
print(
    f"  - Shapiro-Wilk (TA) : W = {stat_s_ta_h:.4f}, p = {p_s_ta_h:.4f} -> {'충족' if norm_ta_h else '위반'}"
)
print(
    f"  - Shapiro-Wilk (Fa) : W = {stat_s_fa_h:.4f}, p = {p_s_fa_h:.4f} -> {'충족' if norm_fa_h else '위반'}"
)
print(
    f"  - Levene 등분산성   : F = {stat_l_h:.4f}, p = {p_l_h:.4f} -> {'등분산 충족' if equal_var_h else '등분산 위반'}"
)
print("-" * 65)
print(f"4. 검정 기법 선택 및 최종 결과:")
print(f"  - 선택된 검정 기법: {test_method_2}")
print(f"  - t-통계량 (t-statistic): {t_stat_2:.4f}")
print(f"  - p-값 (p-value)        : {p_val_2:.4f}")
print(
    f"  - 통계적 유의성 판정    : {'[유의한 차이 있음] (p < 0.05)' if p_val_2 < 0.05 else '[유의한 차이 없음] (p >= 0.05)'}"
)
print("=" * 65)

[필수 2: HeatingQC 'TA' vs 'Fa' 분석 결과]
1. 집단 관계: 서로 다른 주택들이므로 [독립표본 (Independent)]
2. 표본 통계치:
  - TA (Typical): n = 20, 표본평균 = $130,845.00
  - Fa (Fair)   : n = 20, 표본평균 = $122,855.00
  - 평균 차이   : $7,990.00
-----------------------------------------------------------------
3. 사전 가정 검정 (α = 0.05):
  - Shapiro-Wilk (TA) : W = 0.9736, p = 0.8290 -> 충족
  - Shapiro-Wilk (Fa) : W = 0.9319, p = 0.1681 -> 충족
  - Levene 등분산성   : F = 1.3104, p = 0.2595 -> 등분산 충족
-----------------------------------------------------------------
4. 검정 기법 선택 및 최종 결과:
  - 선택된 검정 기법: 독립표본 t-검정 (Student's t-test, equal_var=True)
  - t-통계량 (t-statistic): 0.5584
  - p-값 (p-value)        : 0.5799
  - 통계적 유의성 판정    : [유의한 차이 없음] (p >= 0.05)


### 필수 2 답변 작성란

- **Q1. 두 집단은 독립집단인가요, 대응집단인가요?**
  * **답변:** **독립집단(Independent samples)**입니다. 난방 품질이 'TA'인 주택 집단과 'Fa'인 주택 집단은 서로 다른 개별 주택들로 구성되어 있어 상호 간에 물리적·구조적 종속 관계가 전혀 없기 때문입니다.
- **Q2. 두 집단의 정규성 가정은 충족되나요?**
  * **답변:** **네, 충족됩니다.** Shapiro-Wilk 검정 결과 두 집단 모두 $p$-value가 $0.05$보다 크므로($p \ge 0.05$) 정규분포를 따른다는 귀무가설을 기각할 수 없습니다.
- **Q3. 두 집단의 등분산성 가정은 충족되나요?**
  * **답변:** **네, 충족됩니다.** Levene 검정 결과 $p$-value가 $0.05$보다 크므로($p \ge 0.05$) 두 집단의 분산이 동일하다는 귀무가설을 기각할 수 없습니다.
- **Q4. 가정 점검 결과 어떤 t검정 방법을 선택해야 하나요?**
  * **답변:** 두 집단 모두 정규성을 만족하고 등분산성도 충족하였으므로, 전통적인 **독립표본 t-검정(Student's t-test, `equal_var=True`)**을 선택해야 합니다.
- **Q5. 최종 검정 결과는 무엇을 의미하나요?**
  * **답변:** t-검정 결과 $p$-value가 유의수준 $0.05$보다 작다면($p < 0.05$) 난방 품질이 Typical('TA')인 주택과 Fair('Fa')인 주택 간 판매가격에 **통계적으로 유의미한 차이가 존재함**을 의미합니다.

---

## 과제. 난방 품질에 따른 t검정 방법 선택

### 문제 3-1. 난방 품질 `Ex`와 `TA` 집단의 판매가격 비교

#### 문제 설명

난방 품질이 `Ex`(Excellent)인 주택과 `TA`(Typical/Average)인 주택의 판매가격을 비교하려고 합니다. 각 집단에서 20개씩 표본을 추출한 뒤 필수 문제에서 학습한 **독립표본 t검정과 Welch t검정 선택 과정**을 독립적으로 적용하세요.

#### 요구사항

1. `HeatingQC == "Ex"`인 집단과 `HeatingQC == "TA"`인 집단에서 `SalePrice`를 20개씩 추출하세요.
2. 두 집단의 표본 수와 평균을 출력하세요.
3. 두 집단이 독립집단인지 대응집단인지 판단하세요.
4. 각 집단의 정규성과 두 집단의 등분산성을 검정하세요.
5. 두 집단 모두 정규성을 충족하는지 확인하세요.
6. 등분산성 결과에 따라 독립표본 t검정 또는 Welch t검정 중 적절한 방법을 선택하고 선택 이유를 작성하세요.
7. 선택한 검정을 실행하여 검정통계량과 p-value를 출력하세요.
8. 난방 품질에 따라 판매가격에 유의한 차이가 있는지 결론을 작성하세요.

#### 해석 질문

**Q1.** 두 집단의 정규성 가정은 충족되나요?  
**Q2.** 두 집단의 등분산성 가정은 충족되나요?  
**Q3.** 최종적으로 어떤 t검정 방법을 선택해야 하나요?  
**Q4.** 검정 결과 난방 품질에 따른 판매가격 차이는 통계적으로 유의한가요?

#### 제출 결과

- 표본 구성과 집단 관계 판단
- 정규성 및 등분산성 검정 결과
- 최종 t검정 선택과 근거
- 검정통계량과 p-value
- 결과 해석
- Q1~Q4 답변


In [4]:
# [과제] 난방 품질 'Ex'와 'TA'의 판매가격 비교 및 의사결정 파이프라인

# 1. 20개씩 추출 (random_state=42)
group_ex_heat = (
    df[df["HeatingQC"] == "Ex"]["SalePrice"]
    .dropna()
    .sample(n=20, random_state=42)
)
group_ta_heat = (
    df[df["HeatingQC"] == "TA"]["SalePrice"]
    .dropna()
    .sample(n=20, random_state=42)
)

# 2. 표본 통계치
n_ex, mean_ex = len(group_ex_heat), group_ex_heat.mean()
n_ta, mean_ta = len(group_ta_heat), group_ta_heat.mean()

# 4. 정규성 및 등분산성 검정
stat_s_ex, p_s_ex = stats.shapiro(group_ex_heat)
stat_s_ta, p_s_ta = stats.shapiro(group_ta_heat)
stat_l_exta, p_l_exta = stats.levene(group_ex_heat, group_ta_heat)

norm_ex = p_s_ex >= 0.05
norm_ta = p_s_ta >= 0.05
equal_var_exta = p_l_exta >= 0.05

# 6. t-검정 기법 선택
if norm_ex and norm_ta:
    if equal_var_exta:
        chosen_test = "독립표본 t-검정 (Student's t-test, equal_var=True)"
        t_stat_hw, p_val_hw = stats.ttest_ind(
            group_ex_heat, group_ta_heat, equal_var=True
        )
    else:
        chosen_test = "웰치 t-검정 (Welch's t-test, equal_var=False)"
        t_stat_hw, p_val_hw = stats.ttest_ind(
            group_ex_heat, group_ta_heat, equal_var=False
        )
else:
    # Welch t검정은 등분산성 가정을 완화하지만 비정규성을 해결하는 방법은 아닙니다.
    # 정규성 위반 신호가 있으면 분포·이상치·표본 크기를 추가 점검하고
    # 필요하면 변환, 비모수 검정 또는 강건한 방법을 검토합니다.
    chosen_test = "정규성 가정 추가 점검 필요 (비모수/강건 방법 검토)"
    t_stat_hw, p_val_hw = np.nan, np.nan

# 7 & 8. 결과 출력
print("=" * 65)
print("[과제: HeatingQC 'Ex' vs 'TA' 분석 결과]")
print("=" * 65)
print(f"1. 집단 관계: 서로 다른 주택 그룹 -> [독립표본 (Independent)]")
print(f"2. 표본 통계:")
print(f"  - Ex (Excellent): n = {n_ex}, 평균 = ${mean_ex:,.2f}")
print(f"  - TA (Typical)  : n = {n_ta}, 평균 = ${mean_ta:,.2f}")
print(f"  - 평균 가격 차이: ${mean_ex - mean_ta:,.2f}")
print("-" * 65)
print(f"3. 사전 가정 점검 (α = 0.05):")
print(
    f"  - Shapiro-Wilk (Ex) : W = {stat_s_ex:.4f}, p = {p_s_ex:.4f} -> {'정규성 위반 증거 없음' if norm_ex else '정규성 위반 신호'}"
)
print(
    f"  - Shapiro-Wilk (TA) : W = {stat_s_ta:.4f}, p = {p_s_ta:.4f} -> {'정규성 위반 증거 없음' if norm_ta else '정규성 위반 신호'}"
)
print(
    f"  - Levene 등분산성   : F = {stat_l_exta:.4f}, p = {p_l_exta:.4f} -> {'등분산 충족' if equal_var_exta else '등분산 위반'}"
)
print("-" * 65)
print(f"4. 최종 t-검정 선택 및 결과:")
print(f"  - 선택 기법: {chosen_test}")
print(f"  - t-통계량 : {t_stat_hw:.4f}")
print(f"  - p-값     : {p_val_hw:.4e}")
if np.isnan(p_val_hw):
    print("  - 최종 결론: 정규성 가정을 추가 점검한 뒤 검정 방법을 결정합니다.")
else:
    print(
        f"  - 최종 결론: {'[유의한 가격 차이 있음] (p < 0.05)' if p_val_hw < 0.05 else '[유의한 차이 없음]'}"
    )
print("=" * 65)

[과제: HeatingQC 'Ex' vs 'TA' 분석 결과]
1. 집단 관계: 서로 다른 주택 그룹 -> [독립표본 (Independent)]
2. 표본 통계:
  - Ex (Excellent): n = 20, 평균 = $259,451.20
  - TA (Typical)  : n = 20, 평균 = $130,845.00
  - 평균 가격 차이: $128,606.20
-----------------------------------------------------------------
3. 사전 가정 점검 (α = 0.05):
  - Shapiro-Wilk (Ex) : W = 0.9447, p = 0.2939 -> 정규성 위반 증거 없음
  - Shapiro-Wilk (TA) : W = 0.9736, p = 0.8290 -> 정규성 위반 증거 없음
  - Levene 등분산성   : F = 6.9271, p = 0.0122 -> 등분산 위반
-----------------------------------------------------------------
4. 최종 t-검정 선택 및 결과:
  - 선택 기법: 웰치 t-검정 (Welch's t-test, equal_var=False)
  - t-통계량 : 4.9073
  - p-값     : 4.9458e-05
  - 최종 결론: [유의한 가격 차이 있음] (p < 0.05)


### 과제 답변 작성란

- **Q1. 두 집단의 정규성 가정은 충족되나요?**
  * **답변:** Shapiro-Wilk 검정 결과 Ex 집단($p=0.2939$)과 TA 집단($p=0.8290$) 모두 $p \ge 0.05$이므로 **정규성 가정을 위반한다는 통계적 증거를 찾지 못했습니다.** 이는 표본이 정규분포임을 증명한 것은 아니며, 이번 과제에서는 t검정 적용을 막는 뚜렷한 정규성 위반 신호가 없다고 해석합니다.
- **Q2. 두 집단의 등분산성 가정은 충족되나요?**
  * **답변:** **등분산 가정은 기각됩니다.** Levene 검정 결과 $p=0.0122<0.05$이므로 두 집단의 분산이 같다는 귀무가설을 기각합니다.
- **Q3. 최종적으로 어떤 t검정 방법을 선택해야 하나요?**
  * **답변:** **웰치 t-검정(Welch's t-test, `equal_var=False`)**을 선택합니다. 정규성 위반의 뚜렷한 증거는 없지만 등분산 가정이 기각되었으므로, 두 집단의 분산이 같다고 전제하지 않는 Welch 검정이 적절합니다. Welch 검정은 등분산성 문제를 다루는 방법이며 비정규성 자체를 해결하는 검정은 아닙니다.
- **Q4. 검정 결과 난방 품질에 따른 판매가격 차이는 통계적으로 유의한가요?**
  * **답변:** **네, 통계적으로 유의합니다.** Welch t검정 결과 $t=4.9073$, $p\approx4.95\times10^{-5}$로 유의수준 0.05보다 작아 두 집단의 평균 판매가격이 같다는 귀무가설을 기각합니다. 다만 관측자료이므로 이 결과만으로 난방 품질 자체가 판매가격 차이를 유발했다고 결론 내릴 수는 없습니다.

## 5. 실습 마무리 답변

1. **두 집단을 비교하기 전에 어떤 데이터 특성을 먼저 확인해야 하나요?**
   * **답변:**
     * **데이터의 척도/유형**: 비교 대상이 되는 종속변수가 연속형 수치인지 확인합니다.
     * **집단의 관계**: 서로 다른 대상 간의 비교인지(**독립표본**), 아니면 동일 대상의 전/후 측정인지(**대응표본**) 확인합니다.
     * **데이터의 분포 및 분산**: 표본 크기와 함께 **정규성(Normality)**과 **등분산성(Homogeneity of Variance)** 충족 여부를 확인해야 합니다.

2. **정규성 검정과 등분산 검정에서 `p > 0.05`는 무엇을 의미하나요?**
   * **답변:** 두 검정 모두 **귀무가설($H_0$)이 "가정을 만족한다(정규분포를 따른다 / 분산이 동일하다)"**로 설정되어 있습니다. 따라서 $p > 0.05$라는 것은 귀무가설을 기각할 만한 통계적 증거가 부족하다는 뜻입니다. 즉, **"현재 표본에서 해당 가정을 위반한다는 충분한 증거를 찾지 못했다"**고 해석하며, 가정이 참임을 증명한 것은 아닙니다.

3. **두 집단 모두 정규성을 충족하고 등분산성도 충족하면 어떤 검정을 사용할 수 있나요?**
   * **답변:** 두 가정이 모두 충족될 경우, 두 집단의 분산을 통합(Pooled)하여 계산하는 전통적인 **독립표본 t-검정(Student's t-test, `equal_var=True`)**을 사용합니다.

4. **두 집단 모두 정규성을 충족하지만 등분산성이 위반되면 어떤 검정을 사용할 수 있나요?**
   * **답변:** 두 집단의 분산이 서로 다름을 인정하고 자유도(Degrees of Freedom)를 통계적으로 보정하여 제1종 오류를 통제하는 **웰치 t-검정(Welch's t-test, `equal_var=False`)**을 사용합니다.

5. **독립표본 t검정과 Welch t검정을 선택할 때 정규성과 등분산성을 함께 확인해야 하는 이유는 무엇인가요?**
   * **답변:** 가설검정 기법은 수학적으로 특정 전제조건(가정) 위에서 유도된 확률분포를 사용합니다. 등분산 가정이 위반되었는데도 일반 독립표본 t검정을 무리하게 적용하면 **제1종 오류(실제로는 차이가 없는데 차이가 있다고 판정하는 위양성)의 발생 확률이 설정한 유의수준($\alpha=0.05$)보다 비정상적으로 치솟기 때문**입니다. 따라서 분포 형태와 이상치, 표본 크기를 함께 살피고, t검정 적용이 합리적인 상황에서는 등분산성 여부에 따라 Student t검정과 Welch t검정을 구분해야 합니다.